In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from scipy.stats import wilcoxon, spearmanr
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import NearMiss
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# Project paths
# ============================================================================
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODEL_DIR = RESULTS_DIR / 'model'
TABLE_DIR = RESULTS_DIR / 'table'

print("Project paths ready")

print("=" * 80)
print("Ablation Study: CF Selection Strategy Comparison (Same CF Pool)")
print("  Condition A: No immutable constraint  + Random selection")
print("  Condition B: Immutable constraint     + Random selection (first valid)")
print("  Condition C: Immutable constraint     + Quality Score selection [Proposed]")
print("=" * 80)

# ============================================================================
# 1. Load Data
# ============================================================================
cf_all = pd.read_csv(DATA_DIR / 'cf_results_4industry.csv')
model = joblib.load(MODEL_DIR / 'base_model_final_full.pkl')
features = joblib.load(MODEL_DIR / 'selected_features_final_full.pkl')
threshold = joblib.load(MODEL_DIR / 'base_model_threshold_final_full.pkl')

df_orig = pd.read_csv(DATA_DIR / 'selected_data_for_modeling_full_ratio_clean.csv')
X_full = df_orig[features]
y_full = df_orig['PERF_12M']

nm = NearMiss(version=1, n_neighbors=3)
X_resampled, y_resampled = nm.fit_resample(X_full, y_full)

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled,
    test_size=0.2, random_state=42, stratify=y_resampled
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)

print(f"Scaler fitted on X_train (n={len(X_train)}).")

solvent_X_raw = df_orig[df_orig['PERF_12M'] == 0][features]
solvent_X_scaled = scaler.transform(solvent_X_raw)

iso = IsolationForest(n_estimators=100, contamination=0.1, random_state=42, n_jobs=-1)
iso.fit(solvent_X_scaled)

print(f"Isolation Forest fitted on solvent firms (n={len(solvent_X_raw)}, scaled space).")
print(f"\nTotal CF candidates : {len(cf_all)} rows")
print(f"Unique firms        : {cf_all['ID'].nunique()}")
print(f"Threshold           : {threshold:.4f}")

# ============================================================================
# 2. Define Immutable Features
# ============================================================================
IMMUTABLE = [
    'asset_growth_rate', 'revenue_growth_rate', 'operating_income_growth',
    'net_income_growth', 'equity_growth_rate',
]

missing_immutables = [f for f in IMMUTABLE if f not in features]
if missing_immutables:
    raise ValueError(f"Immutable features not found: {missing_immutables}")

# ============================================================================
# 3. Metric Computation Functions
# ============================================================================

def get_cf_feature_array(row):
    return np.array([row[f'CF_{f}'] for f in features])

def get_orig_feature_array(row):
    return np.array([row[f'Original_{f}'] for f in features])

def compute_validity(row, model, threshold):
    cf_arr = get_cf_feature_array(row).reshape(1, -1)
    prob = model.predict_proba(cf_arr)[0, 1]
    return 1.0 if prob < threshold else 0.0

def compute_proximity(row, scaler):
    orig = get_orig_feature_array(row)
    cf = get_cf_feature_array(row)
    orig_scaled = scaler.transform(orig.reshape(1, -1))[0]
    cf_scaled = scaler.transform(cf.reshape(1, -1))[0]
    return float(np.mean(np.abs(orig_scaled - cf_scaled)))

def compute_sparsity(row, scaler, tol=1e-6):
    orig = get_orig_feature_array(row)
    cf = get_cf_feature_array(row)
    orig_scaled = scaler.transform(orig.reshape(1, -1))[0]
    cf_scaled = scaler.transform(cf.reshape(1, -1))[0]
    changes = np.abs(orig_scaled - cf_scaled)
    return float(np.mean(changes < tol))

def compute_realism(row, scaler, iso_model):
    cf_arr = get_cf_feature_array(row).reshape(1, -1)
    cf_scaled = scaler.transform(cf_arr)
    iso_score = iso_model.decision_function(cf_scaled)[0]
    return float(iso_score + 0.5) if iso_score > -0.5 else 0.0

def compute_robustness(row, model, threshold, n=10, noise_level=0.01):
    cf_arr = get_cf_feature_array(row)
    count = 0
    for _ in range(n):
        noise = np.random.normal(0, noise_level, size=cf_arr.shape)
        perturbed = cf_arr + (noise * (cf_arr + 1e-6))
        if model.predict_proba(perturbed.reshape(1, -1))[0, 1] < threshold:
            count += 1
    return count / n

def check_immutable_violated(row, tol=1e-3):
    """
    Returns True only if an immutable growth-rate feature changed by more
    than a financially meaningful amount. tol=1e-3 was chosen after
    inspecting borderline cases: DiCE's genetic algorithm introduces
    floating-point noise on the order of 1e-6 even when a feature was
    never targeted for change (verified on 11 firms out of 829, where all
    observed "changes" were ~1e-6 to 2e-6 — e.g. operating_income_growth
    39.028413 -> 39.028412), which a stricter 1e-6 threshold incorrectly
    flagged as violations.
    """
    for f in IMMUTABLE:
        if abs(row[f'Original_{f}'] - row[f'CF_{f}']) > tol:
            return True
    return False

def compute_quality_score(v, prox, sp, real, rob):
    return (5 * v + (1 - prox) + sp + real + rob) / 9

# ============================================================================
# 4. Pre-compute Metrics for All CF Candidates
# ============================================================================
print("\n[Step 1] Computing metrics for all CF candidates...")

records = []
for _, row in cf_all.iterrows():
    v = compute_validity(row, model, threshold)
    prox = compute_proximity(row, scaler)
    sp = compute_sparsity(row, scaler)
    real = compute_realism(row, scaler, iso)
    rob = compute_robustness(row, model, threshold)
    qs = compute_quality_score(v, prox, sp, real, rob)
    immut_violated = check_immutable_violated(row)

    records.append({
        'ID': row['ID'], 'SIC_CD_3': row.get('SIC_CD_3', np.nan),
        'CF_Number': row['CF_Number'], 'Validity': v, 'Proximity': prox,
        'Sparsity': sp, 'Realism': real, 'Robustness': rob,
        'QualityScore': qs, 'Immutable_Violated': immut_violated,
    })

metrics_df = pd.DataFrame(records)
print(f"Done. Total rows: {len(metrics_df)}")
print(f"Rows with immutable violation (tol=1e-3): {metrics_df['Immutable_Violated'].sum()} "
      f"({metrics_df['Immutable_Violated'].mean()*100:.1f}%)")

# ============================================================================
# 5. Select Best CF per Firm under Each Condition (per-firm seed, from prior fix)
# ============================================================================
print("\n[Step 2] Selecting best CF per firm under each condition...")

ids = cf_all['ID'].unique()
results = {'A': [], 'B': [], 'C': []}

for firm_id in ids:
    firm = metrics_df[metrics_df['ID'] == firm_id].copy()
    firm_seed = int(firm_id) % (2**31 - 1)

    pool_A = firm
    sel_A = pool_A.sample(1, random_state=firm_seed).iloc[0]
    results['A'].append(sel_A)

    pool_B = firm[~firm['Immutable_Violated']]
    if pool_B.empty:
        pool_B = firm
    sel_B = pool_B.sample(1, random_state=firm_seed).iloc[0]
    results['B'].append(sel_B)

    pool_C = firm[~firm['Immutable_Violated']]
    if pool_C.empty:
        pool_C = firm
    sel_C = pool_C.loc[pool_C['QualityScore'].idxmax()]
    results['C'].append(sel_C)

df_A = pd.DataFrame(results['A'])
df_B = pd.DataFrame(results['B'])
df_C = pd.DataFrame(results['C'])

n_firms = len(ids)

# ============================================================================
# 5b. Verification — how many firms now fall back to pool_A due to all-4 violation?
# ============================================================================
firms_with_violation = metrics_df[metrics_df['Immutable_Violated']]['ID'].unique()
firms_all_four_violated = []
for firm_id in firms_with_violation:
    firm = metrics_df[metrics_df['ID'] == firm_id]
    if firm['Immutable_Violated'].sum() == len(firm):
        firms_all_four_violated.append(firm_id)

diff_count = 0
for firm_id in firms_with_violation:
    a_choice = df_A[df_A['ID'] == firm_id]['CF_Number'].values
    b_choice = df_B[df_B['ID'] == firm_id]['CF_Number'].values
    if len(a_choice) > 0 and len(b_choice) > 0 and a_choice[0] != b_choice[0]:
        diff_count += 1

print(f"\n[Verification] Firms with a genuine (tol=1e-3) immutable violation: {len(firms_with_violation)}")
print(f"[Verification] Of these, firms where ALL 4 CFs violated (pool_B falls back to pool_A): {len(firms_all_four_violated)}")
print(f"[Verification] Of these, A and B select a DIFFERENT CF: {diff_count} / {len(firms_with_violation)}")

# ============================================================================
# 6. Summarise Results
# ============================================================================
METRICS = ['Validity', 'Proximity', 'Sparsity', 'Realism', 'Robustness', 'QualityScore']

print("\n" + "=" * 80)
print(f"ABLATION STUDY RESULTS (n={n_firms} firms each condition, 4-industry subset)")
print("=" * 80)

summary = {}
for cond, df in [('A (No constraint + Random)', df_A),
                 ('B (Immutable + Random)', df_B),
                 ('C (Immutable + QScore) [Proposed]', df_C)]:
    row = {}
    for m in METRICS:
        row[m] = f"{df[m].mean():.4f} ± {df[m].std():.4f}"
    summary[cond] = row

summary_df = pd.DataFrame(summary).T
print(summary_df.to_string())

# ============================================================================
# 7. Statistical Tests: A vs C and B vs C (Wilcoxon signed-rank)
# ============================================================================
print("\n[Step 3] Statistical Tests (Wilcoxon signed-rank, A vs C and B vs C)")
print("-" * 60)

test_results = []
for m in ['Proximity', 'Sparsity', 'Realism', 'QualityScore']:
    stat_ac, p_ac = wilcoxon(df_A[m], df_C[m])
    stat_bc, p_bc = wilcoxon(df_B[m], df_C[m])

    r_ac = 1 - (2 * stat_ac) / (n_firms * (n_firms + 1))
    r_bc = 1 - (2 * stat_bc) / (n_firms * (n_firms + 1))

    print(f"\n{m}:")
    print(f"  A vs C -> p={p_ac:.4f}, effect r={r_ac:.3f} "
          f"{'***' if p_ac<0.001 else '**' if p_ac<0.01 else '*' if p_ac<0.05 else 'ns'}")
    print(f"  B vs C -> p={p_bc:.4f}, effect r={r_bc:.3f} "
          f"{'***' if p_bc<0.001 else '**' if p_bc<0.01 else '*' if p_bc<0.05 else 'ns'}")

    test_results.append({'metric': m, 'p_AC': p_ac, 'effect_r_AC': r_ac,
                          'p_BC': p_bc, 'effect_r_BC': r_bc})

# ============================================================================
# 8. Sensitivity Analysis: Quality Score Weight Schemes
# ============================================================================
print("\n[Step 4] Sensitivity Analysis: Weight Scheme Comparison")
print("-" * 60)

weight_schemes = {
    'Proposed (5,1,1,1,1)': (5, 1, 1, 1, 1),
    'Equal (1,1,1,1,1)': (1, 1, 1, 1, 1),
    'Prox-heavy (5,3,1,1,1)': (5, 3, 1, 1, 1),
    'Real-heavy (5,1,1,3,1)': (5, 1, 1, 3, 1),
    'Rob-heavy (5,1,1,1,3)': (5, 1, 1, 1, 3),
}

firm_ids_list = list(ids)
scheme_ranks = {s: [] for s in weight_schemes if s != 'Proposed (5,1,1,1,1)'}

for firm_id in firm_ids_list:
    firm = metrics_df[(metrics_df['ID'] == firm_id) & (~metrics_df['Immutable_Violated'])].copy()
    if firm.empty:
        firm = metrics_df[metrics_df['ID'] == firm_id].copy()

    denom_proposed = sum(weight_schemes['Proposed (5,1,1,1,1)'])
    firm['QS_proposed'] = (
        weight_schemes['Proposed (5,1,1,1,1)'][0] * firm['Validity'] +
        weight_schemes['Proposed (5,1,1,1,1)'][1] * (1 - firm['Proximity']) +
        weight_schemes['Proposed (5,1,1,1,1)'][2] * firm['Sparsity'] +
        weight_schemes['Proposed (5,1,1,1,1)'][3] * firm['Realism'] +
        weight_schemes['Proposed (5,1,1,1,1)'][4] * firm['Robustness']
    ) / denom_proposed

    best_proposed_cfnum = firm.loc[firm['QS_proposed'].idxmax(), 'CF_Number']

    for scheme_name, (wv, wp, ws, wr, wrob) in weight_schemes.items():
        if scheme_name == 'Proposed (5,1,1,1,1)':
            continue
        denom = wv + wp + ws + wr + wrob
        firm[f'QS_{scheme_name}'] = (
            wv * firm['Validity'] + wp * (1 - firm['Proximity']) +
            ws * firm['Sparsity'] + wr * firm['Realism'] + wrob * firm['Robustness']
        ) / denom
        best_other_cfnum = firm.loc[firm[f'QS_{scheme_name}'].idxmax(), 'CF_Number']
        scheme_ranks[scheme_name].append(int(best_proposed_cfnum == best_other_cfnum))

print(f"\nSelection Agreement Rate with Proposed Scheme (per firm, n={len(firm_ids_list)}):")
agreement_log = []
for scheme_name, agreements in scheme_ranks.items():
    rate = np.mean(agreements) * 100
    print(f"  {scheme_name:30s}: {rate:.1f}% agreement")
    agreement_log.append({'scheme': scheme_name, 'agreement_pct': rate})

print(f"\nSpearman Rank Correlation of QualityScore with Proposed Scheme (all {len(metrics_df)} candidates):")
pool_C_metrics = metrics_df[~metrics_df['Immutable_Violated']].copy()
wv0, wp0, ws0, wr0, wrob0 = weight_schemes['Proposed (5,1,1,1,1)']
denom0 = wv0 + wp0 + ws0 + wr0 + wrob0
pool_C_metrics['QS_proposed'] = (
    wv0 * pool_C_metrics['Validity'] + wp0 * (1 - pool_C_metrics['Proximity']) +
    ws0 * pool_C_metrics['Sparsity'] + wr0 * pool_C_metrics['Realism'] +
    wrob0 * pool_C_metrics['Robustness']
) / denom0

spearman_log = []
for scheme_name, (wv, wp, ws, wr, wrob) in weight_schemes.items():
    if scheme_name == 'Proposed (5,1,1,1,1)':
        continue
    denom = wv + wp + ws + wr + wrob
    qs_other = (
        wv * pool_C_metrics['Validity'] + wp * (1 - pool_C_metrics['Proximity']) +
        ws * pool_C_metrics['Sparsity'] + wr * pool_C_metrics['Realism'] +
        wrob * pool_C_metrics['Robustness']
    ) / denom
    rho, pval = spearmanr(pool_C_metrics['QS_proposed'], qs_other)
    print(f"  {scheme_name:30s}: rho = {rho:.4f}, p = {pval:.4f}")
    spearman_log.append({'scheme': scheme_name, 'spearman_rho': rho, 'p_value': pval})

# ============================================================================
# 9. Save All Results
# ============================================================================
df_A.to_csv(DATA_DIR / 'ablation_condA_full.csv', index=False)
df_B.to_csv(DATA_DIR / 'ablation_condB_full.csv', index=False)
df_C.to_csv(DATA_DIR / 'ablation_condC_full.csv', index=False)
metrics_df.to_csv(DATA_DIR / 'ablation_all_metrics_full.csv', index=False)

summary_df.to_csv(TABLE_DIR / 'ablation_summary_full.csv')
pd.DataFrame(test_results).to_csv(TABLE_DIR / 'ablation_wilcoxon_tests_full.csv', index=False)
pd.DataFrame(agreement_log).to_csv(TABLE_DIR / 'ablation_weight_agreement_full.csv', index=False)
pd.DataFrame(spearman_log).to_csv(TABLE_DIR / 'ablation_weight_spearman_full.csv', index=False)

print("\n" + "=" * 80)
print("Ablation study complete. Files saved to data/ and results/table/")
print("=" * 80)

Project paths ready
Ablation Study: CF Selection Strategy Comparison (Same CF Pool)
  Condition A: No immutable constraint  + Random selection
  Condition B: Immutable constraint     + Random selection (first valid)
  Condition C: Immutable constraint     + Quality Score selection [Proposed]
Scaler fitted on X_train (n=3312).
Isolation Forest fitted on solvent firms (n=145654, scaled space).

Total CF candidates : 2443 rows
Unique firms        : 640
Threshold           : 0.4683

[Step 1] Computing metrics for all CF candidates...
Done. Total rows: 2443
Rows with immutable violation (tol=1e-3): 0 (0.0%)

[Step 2] Selecting best CF per firm under each condition...

[Verification] Firms with a genuine (tol=1e-3) immutable violation: 0
[Verification] Of these, firms where ALL 4 CFs violated (pool_B falls back to pool_A): 0
[Verification] Of these, A and B select a DIFFERENT CF: 0 / 0

ABLATION STUDY RESULTS (n=640 firms each condition, 4-industry subset)
                                   